In [ ]:
print("Hello")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install torch torchvision torchaudio --quiet
!pip install opencv-python tqdm scikit-image matplotlib --quiet

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
import os
import matplotlib.pyplot as plt
from skimage.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch ver: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
class ConvBNReLU(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = ConvBNReLU(channels, channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, 1, 1)
        self.bn = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.bn(x)
        x = x + residual
        x = self.relu(x)
        return x

class Encoder(nn.Module):
    def __init__(self, in_channels=4):
        super().__init__()
        self.conv1 = ConvBNReLU(in_channels, 32, kernel_size=3, stride=1, padding=1)
        self.down1 = ConvBNReLU(32, 64, kernel_size=3, stride=2, padding=1)
        self.down2 = ConvBNReLU(64, 128, kernel_size=3, stride=2, padding=1)
        self.down3 = ConvBNReLU(128, 256, kernel_size=3, stride=2, padding=1)
        self.down4 = ConvBNReLU(256, 512, kernel_size=3, stride=2, padding=1)

        self.res1 = ResidualBlock(64)
        self.res2 = ResidualBlock(128)
        self.res3 = ResidualBlock(256)
        self.res4 = ResidualBlock(512)

    def forward(self, x):
        features = {}
        x = self.conv1(x)
        x = self.down1(x)
        x = self.res1(x)
        features['level1'] = x
        x = self.down2(x)
        x = self.res2(x)
        features['level2'] = x
        x = self.down3(x)
        x = self.res3(x)
        features['level3'] = x
        x = self.down4(x)
        x = self.res4(x)
        features['level4'] = x
        return features

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.up4 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBNReLU(512, 256)
        )
        self.conv4 = ResidualBlock(256)

        self.up3 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBNReLU(256, 128)
        )
        self.conv3 = ResidualBlock(128)

        self.up2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBNReLU(128, 64)
        )
        self.conv2 = ResidualBlock(64)

        self.up1 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBNReLU(64, 32)
        )
        self.conv1 = ResidualBlock(32)

        self.alpha_out = nn.Sequential(
            ConvBNReLU(32, 16),
            nn.Conv2d(16, 1, 3, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, encoder_features):
        x = encoder_features['level4']
        x = self.up4(x)
        x = x + encoder_features['level3']
        x = self.conv4(x)
        x = self.up3(x)
        x = x + encoder_features['level2']
        x = self.conv3(x)
        x = self.up2(x)
        x = x + encoder_features['level1']
        x = self.conv2(x)
        x = self.up1(x)
        x = self.conv1(x)
        alpha = self.alpha_out(x)
        return alpha

class SemanticRefineNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder(in_channels=4)
        self.decoder = Decoder()

    def forward(self, rgb, base_alpha):
        x = torch.cat([rgb, base_alpha], dim=1)
        features = self.encoder(x)
        semantic_alpha = self.decoder(features)
        return semantic_alpha

In [ ]:
def load_rvm_model(model_type="resnet50"):
    model = torch.hub.load("PeterL1n/RobustVideoMatting", model_type, pretrained=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    return model, device

def load_your_model(model_path, device):
    model = SemanticRefineNet().to(device)
    checkpoint = torch.load(model_path, map_location=device)

    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        epoch = checkpoint.get('epoch', 'unknown')
        val_loss = checkpoint.get('val_loss', 'unknown')
        print(f"Load model success. Epoch: {epoch}, Val Loss: {val_loss}")
    else:
        model.load_state_dict(checkpoint)
        print(f"Load model success.")

    model.eval()
    return model

def refine_alpha(rvm_alpha, semantic_alpha, delta=1.0, transition_range=(0.05, 0.95)):
    low, high = transition_range
    weight = np.zeros_like(rvm_alpha)
    transition_mask = (rvm_alpha > low) & (rvm_alpha < high)
    weight[transition_mask] = delta
    weight = cv2.GaussianBlur(weight, (5, 5), 1.0)
    refined = (1 - weight) * rvm_alpha + weight * semantic_alpha
    refined = np.clip(refined, 0, 1)
    return refined

In [ ]:
def process_single_frame(image_bgr, rvm_model, semantic_model, device,
                         target_size=(512, 512), delta=1.0):
    
    h, w = image_bgr.shape[:2]

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    img_tensor = torch.from_numpy(image_rgb).permute(2, 0, 1).unsqueeze(0).float() / 255.0
    img_tensor = img_tensor.to(device)

    with torch.no_grad():
        fgr, pha, *rec = rvm_model(img_tensor, None, None)
        rvm_alpha = pha.squeeze().cpu().numpy()
        rvm_alpha = np.clip(rvm_alpha, 0, 1)

    img_resized = cv2.resize(image_rgb, target_size)
    base_resized = cv2.resize(rvm_alpha, target_size)

    img_tensor = torch.from_numpy(img_resized).permute(2, 0, 1).unsqueeze(0).float() / 255.0
    base_tensor = torch.from_numpy(base_resized).unsqueeze(0).unsqueeze(0).float()
    img_tensor = img_tensor.to(device)
    base_tensor = base_tensor.to(device)

    with torch.no_grad():
        semantic_alpha = semantic_model(img_tensor, base_tensor)
        semantic_alpha = semantic_alpha.squeeze().cpu().numpy()
        semantic_alpha = cv2.resize(semantic_alpha, (w, h))
        semantic_alpha = np.clip(semantic_alpha, 0, 1)

    refined_alpha = refine_alpha(rvm_alpha, semantic_alpha, delta=delta)

    return rvm_alpha, semantic_alpha, refined_alpha


def composite_with_bg(image_bgr, alpha, bg_color=(0, 255, 0)):
    h, w = image_bgr.shape[:2]
    if alpha.shape[:2] != (h, w):
        alpha = cv2.resize(alpha, (w, h))

    bg = np.full((h, w, 3), bg_color, dtype=np.uint8)
    alpha_3ch = np.stack([alpha, alpha, alpha], axis=2)

    composite = (image_bgr * alpha_3ch + bg * (1 - alpha_3ch)).astype(np.uint8)
    return composite


def create_output_frame(image_bgr, rvm_alpha, refined_alpha, output_type="both"):
    h, w = image_bgr.shape[:2]

    if output_type == "rvm":
        rvm_composite = composite_with_bg(image_bgr, rvm_alpha)
        return rvm_composite

    elif output_type == "refined":
        refined_composite = composite_with_bg(image_bgr, refined_alpha)
        return refined_composite

    elif output_type == "both":
        rvm_composite = composite_with_bg(image_bgr, rvm_alpha)
        refined_composite = composite_with_bg(image_bgr, refined_alpha)
        return np.hstack([rvm_composite, refined_composite])

    elif output_type == "full":
        input_frame = image_bgr 

        rvm_alpha_vis = (rvm_alpha * 255).astype(np.uint8)
        rvm_alpha_color = cv2.cvtColor(rvm_alpha_vis, cv2.COLOR_GRAY2BGR)
        rvm_composite = composite_with_bg(image_bgr, rvm_alpha)

        refined_alpha_vis = (refined_alpha * 255).astype(np.uint8)
        refined_alpha_color = cv2.cvtColor(refined_alpha_vis, cv2.COLOR_GRAY2BGR)
        refined_composite = composite_with_bg(image_bgr, refined_alpha)

        alpha_diff = np.abs(refined_alpha - rvm_alpha)
        alpha_diff_vis = (alpha_diff * 255).astype(np.uint8)
        alpha_diff_color = cv2.applyColorMap(alpha_diff_vis, cv2.COLORMAP_JET)

        row1 = np.hstack([input_frame, rvm_alpha_color, rvm_composite])
        row2 = np.hstack([refined_alpha_color, refined_composite, alpha_diff_color])
        return np.vstack([row1, row2])

    else:
        raise ValueError(f"Unknown output_type: {output_type}")

In [ ]:
def inference_video(input_video_path, output_video_path, rvm_model, semantic_model, device,
                    target_size=(512, 512), delta=1.0, save_frames_dir=None,
                    sample_interval=30, output_type="both"):
    
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        print(f"Cannot open: {input_video_path}")
        return

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"Video information: {width}x{height}, {fps}fps, {total_frames} frames")

    if output_type == "rvm" or output_type == "refined":
        out_width, out_height = width, height
    elif output_type == "both":
        out_width, out_height = width * 2, height
    elif output_type == "full":
        out_width, out_height = width * 3, height * 2
    else:
        raise ValueError(f"Unknown output_type: {output_type}")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (out_width, out_height))

    if save_frames_dir:
        os.makedirs(save_frames_dir, exist_ok=True)

    frame_count = 0

    with tqdm(total=total_frames, desc="Processing") as pbar:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            rvm_alpha, semantic_alpha, refined_alpha = process_single_frame(
                frame, rvm_model, semantic_model, device, target_size, delta
            )

            output_frame = create_output_frame(frame, rvm_alpha, refined_alpha, output_type)

            out.write(output_frame)

            if save_frames_dir and frame_count % sample_interval == 0 and output_type == "full":
                output_rgb = cv2.cvtColor(output_frame, cv2.COLOR_BGR2RGB)
                plt.figure(figsize=(18, 12))
                plt.imshow(output_rgb)
                plt.title(f"Frame {frame_count} | Delta={delta}", fontsize=14)
                plt.axis('off')
                plt.tight_layout()
                plt.savefig(os.path.join(save_frames_dir, f"frame_{frame_count:06d}.png"),
                           dpi=150, bbox_inches='tight')
                plt.close()

            frame_count += 1
            pbar.update(1)
            pbar.set_postfix({'frame': frame_count})

    cap.release()
    out.release()
    print(f"   output path: {output_video_path}")
    print(f"  Frame count: {frame_count}")

In [ ]:
def main():
    INPUT_VIDEO = "/content/drive/MyDrive/DATA/test/input3.mp4"
    OUTPUT_VIDEO = "/content/drive/MyDrive/DATA/test/output3.mp4"
    MODEL_PATH = "/content/drive/MyDrive/RVM/semantic_refine_model/model_best.pth"

    SAVE_FRAMES_DIR = "/content/drive/MyDrive/output_frames"

    OUTPUT_TYPE = "refined"  # "rvm" "refined" "both" "full"

    DELTA = 1.0
    TARGET_SIZE = (512, 512)
    SAMPLE_INTERVAL = 30

    print("=" * 60)
    print("Semantic Refinement")
    print("=" * 60)
    print(f"Input video: {INPUT_VIDEO}")
    print(f"Output video: {OUTPUT_VIDEO}")
    print(f"Refinement δ = {DELTA}")
    print("=" * 60)

    if not os.path.exists(INPUT_VIDEO):
        print(f"Cannot find: {INPUT_VIDEO}")
        return

    rvm_model, device = load_rvm_model(model_type="resnet50")
    semantic_model = load_your_model(MODEL_PATH, device)

    inference_video(
        input_video_path=INPUT_VIDEO,
        output_video_path=OUTPUT_VIDEO,
        rvm_model=rvm_model,
        semantic_model=semantic_model,
        device=device,
        target_size=TARGET_SIZE,
        delta=DELTA,
        save_frames_dir=SAVE_FRAMES_DIR if OUTPUT_TYPE == "full" else None,
        sample_interval=SAMPLE_INTERVAL,
        output_type=OUTPUT_TYPE
    )


if __name__ == "__main__":
    main()

In [ ]:
def test_single_image(image_path, rvm_model, semantic_model, device, delta=1.0):
    image = cv2.imread(image_path)
    if image is None:
        print(f"Cannot open: {image_path}")
        return

    rvm_alpha, semantic_alpha, refined_alpha = process_single_frame(
        image, rvm_model, semantic_model, device, delta=delta
    )

    rvm_composite = composite_with_bg(image, rvm_alpha)
    refined_composite = composite_with_bg(image, refined_alpha)

    alpha_diff = np.abs(refined_alpha - rvm_alpha)

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    axes[0, 0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axes[0, 0].set_title("Input")
    axes[0, 0].axis('off')

    axes[0, 1].imshow(rvm_alpha, cmap='gray')
    axes[0, 1].set_title("RVM Alpha")
    axes[0, 1].axis('off')

    axes[0, 2].imshow(rvm_composite)
    axes[0, 2].set_title("RVM Output")
    axes[0, 2].axis('off')

    axes[1, 0].imshow(refined_alpha, cmap='gray')
    axes[1, 0].set_title(f"Refine Alpha (δ={delta})")
    axes[1, 0].axis('off')

    axes[1, 1].imshow(refined_composite)
    axes[1, 1].set_title("Refine Output")
    axes[1, 1].axis('off')

    axes[1, 2].imshow(alpha_diff, cmap='hot')
    axes[1, 2].set_title("Alpha Diff (RVM - Refine)")
    axes[1, 2].axis('off')

    plt.tight_layout()
    plt.show()

# test_image_path = "/content/drive/MyDrive/test_image.jpg"
# test_single_image(test_image_path, rvm_model, semantic_model, device, delta=1.0)